In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in working directory as specified)
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')

# Convert timestamps to timezone-aware datetimes (UTC -> Asia/Shanghai)
# Keep original numeric timestamp as well
df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_trace['ts_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Metric summary components
metric_row_count = df_metric.shape[0]
metric_min_ts = df_metric['ts_dt'].min()
metric_max_ts = df_metric['ts_dt'].max()
metric_cmdb_ids = df_metric['cmdb_id'].dropna().unique()
# create a compact representation of distinct cmdb_ids (comma-separated, truncated if long)
cmdb_list_str = ", ".join(map(str, metric_cmdb_ids[:50]))
if len(metric_cmdb_ids) > 50:
    cmdb_list_str += ", ...(+%d more)" % (len(metric_cmdb_ids) - 50)

metric_top_kpis = df_metric['kpi_name'].value_counts().head(20).reset_index()
metric_top_kpis.columns = ['kpi_name', 'count']

metric_sample = df_metric[['timestamp', 'ts_dt', 'cmdb_id', 'kpi_name', 'value']].head(5)

metric_info = pd.DataFrame({
    'metric': ['row_count', 'min_timestamp', 'max_timestamp', 'distinct_cmdb_ids_count', 'distinct_cmdb_ids_sample'],
    'value': [metric_row_count, metric_min_ts, metric_max_ts, len(metric_cmdb_ids), cmdb_list_str]
})

# Trace summary components
trace_row_count = df_trace.shape[0]
trace_min_ts = df_trace['ts_dt'].min()
trace_max_ts = df_trace['ts_dt'].max()
trace_cmdb_ids = df_trace['cmdb_id'].dropna().unique()
trace_cmdb_list_str = ", ".join(map(str, trace_cmdb_ids[:50]))
if len(trace_cmdb_ids) > 50:
    trace_cmdb_list_str += ", ...(+%d more)" % (len(trace_cmdb_ids) - 50)

trace_top_names = df_trace['trace_name'].value_counts().head(20).reset_index()
trace_top_names.columns = ['trace_name', 'count']

trace_sample = df_trace[['timestamp', 'ts_dt', 'cmdb_id', 'trace_name', 'value']].head(5)

trace_info = pd.DataFrame({
    'trace': ['row_count', 'min_timestamp', 'max_timestamp', 'distinct_cmdb_ids_count', 'distinct_cmdb_ids_sample'],
    'value': [trace_row_count, trace_min_ts, trace_max_ts, len(trace_cmdb_ids), trace_cmdb_list_str]
})

# Display compact summaries (variables separated by commas as required)
metric_info, metric_top_kpis, metric_sample, trace_info, trace_top_names, trace_sample

```
Out[1]:
```
```python
# Summarize previous execution results in plain English as a compact string.
# Reuse existing variables from the prior successful run: metric_row_count, metric_min_ts, metric_max_ts,
# metric_cmdb_ids, metric_top_kpis, metric_sample, trace_row_count, trace_min_ts, trace_max_ts,
# trace_cmdb_ids, trace_top_names, trace_sample.

# Prepare compact top-5 lists and sample row summaries
metric_top5 = ", ".join(
    f"{r.kpi_name}({int(r.count)})" for r in metric_top_kpis.head(5).itertuples(index=False)
)
trace_top5 = ", ".join(
    f"{r.trace_name}({int(r.count)})" for r in trace_top_names.head(5).itertuples(index=False)
)

metric_cmdb_examples = ", ".join(map(str, list(metric_cmdb_ids)[:5])) + ("..." if len(metric_cmdb_ids) > 5 else "")
trace_cmdb_examples = ", ".join(map(str, list(trace_cmdb_ids)[:5])) + ("..." if len(trace_cmdb_ids) > 5 else "")

def compact_sample(df, cols):
    lines = []
    for row in df.head(5).itertuples(index=False):
        vals = [str(getattr(row, c)) for c in cols]
        lines.append(" | ".join(vals))
    return " ; ".join(lines)

metric_sample_text = compact_sample(metric_sample, ['timestamp', 'ts_dt', 'cmdb_id', 'kpi_name', 'value'])
trace_sample_text = compact_sample(trace_sample, ['timestamp', 'ts_dt', 'cmdb_id', 'trace_name', 'value'])

summary = (
    "Summary of metric.csv and trace.csv:\n\n"
    "metric.csv:\n"
    f"- Total rows: {metric_row_count}\n"
    f"- Time range (Asia/Shanghai): {metric_min_ts} to {metric_max_ts}\n"
    f"- Distinct cmdb_id count: {len(metric_cmdb_ids)} (examples: {metric_cmdb_examples})\n"
    f"- Top kpi_name by count (top 5): {metric_top5}\n"
    f"- First 5 rows (compact): {metric_sample_text}\n\n"
    "trace.csv:\n"
    f"- Total rows: {trace_row_count}\n"
    f"- Time range (Asia/Shanghai): {trace_min_ts} to {trace_max_ts}\n"
    f"- Distinct cmdb_id count: {len(trace_cmdb_ids)} (examples: {trace_cmdb_examples})\n"
    f"- Top trace_name by count (top 5): {trace_top5}\n"
    f"- First 5 rows (compact): {trace_sample_text}\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     metric                                              value
0                 row_count                                              45189
1             min_timestamp                          2020-05-23 00:00:00+08:00
2             max_timestamp                          2020-05-23 00:30:00+08:00
3   distinct_cmdb_ids_count                                                 56
4  distinct_cmdb_ids_sample  db_001, db_002, db_003, db_004, db_005, db_006...,                     kpi_name  count
0                 Send_total    663
1               Disk_io_util    662
2                Disk_rd_ios    661
3       Processor_load_5_min    661
4   Outgoing_network_traffic    661
5   Incoming_network_traffic    661
6                 Recv_total    660
7       Processor_load_1_min    660
8      Processor_load_15_min    660
9              CPU_user_time    660
10                   Page_pi    660
11              CPU_idle_pct    660
12                Agent_ping    660
13             Disk_avgqu_sz    660
14                Disk_svctm    660
15               Memory_used    660
16           Memory_used_pct    660
17          Memory_available    660
18                   Page_po    660
19                  ss_total    660,     timestamp                     ts_dt cmdb_id      kpi_name      value
0  1590163200 2020-05-23 00:00:00+08:00  db_001           ACS   0.010000
1  1590163200 2020-05-23 00:00:00+08:00  db_001          AIOS   0.000000
2  1590163200 2020-05-23 00:00:00+08:00  db_001           AWS   0.020000
3  1590163200 2020-05-23 00:00:00+08:00  db_001  CPU_Used_Pct   1.172419
4  1590163200 2020-05-23 00:00:00+08:00  db_001  CPU_free_pct  98.668513,                       trace                                              value
0                 row_count                                               6240
1             min_timestamp                          2020-05-23 00:00:00+08:00
2             max_timestamp                          2020-05-23 00:29:00+08:00
3   distinct_cmdb_ids_count                                                 10
4  distinct_cmdb_ids_sample  docker_001, docker_002, docker_003, docker_004...,                                        trace_name  count
0                       trace.self.jdbc.row_count    240
1                      trace.self.jdbc.error_rate    240
2                    trace.self.jdbc.duration_p95    240
3                   trace.self.jdbc.duration_mean    240
4                      trace.self.local.row_count    240
5                     trace.self.local.error_rate    240
6                   trace.self.local.duration_p95    240
7                  trace.self.local.duration_mean    240
8                        trace.self.csf.row_count    180
9                       trace.self.csf.error_rate    180
10                    trace.self.csf.duration_p95    180
11                   trace.self.csf.duration_mean    180
12                 trace.self.flyremote.row_count    120
13                trace.self.flyremote.error_rate    120
14              trace.self.flyremote.duration_p95    120
15             trace.self.flyremote.duration_mean    120
16  trace.from_os_022.remoteprocess.duration_mean     60
17   trace.from_os_022.remoteprocess.duration_p95     60
18     trace.from_os_022.remoteprocess.error_rate     60
19      trace.from_os_022.remoteprocess.row_count     60,     timestamp                     ts_dt     cmdb_id                                     trace_name      value
0  1590163200 2020-05-23 00:00:00+08:00  docker_001  trace.from_os_022.remoteprocess.duration_mean   0.739681
1  1590163200 2020-05-23 00:00:00+08:00  docker_001   trace.from_os_022.remoteprocess.duration_p95   2.195850
2  1590163200 2020-05-23 00:00:00+08:00  docker_001     trace.from_os_022.remoteprocess.error_rate   0.000000
3  1590163200 2020-05-23 00:00:00+08:00  docker_001      trace.from_os_022.remoteprocess.row_count  94.000000
4  1590163200 2020-05-23 00:00:00+08:00  docker_001                   trace.self.csf.duration_mean   0.064801)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing dataframes if present (from prior run)
# df_metric, df_trace, tz are expected to exist in the kernel state.

# Define incident window in Asia/Shanghai (UTC+8)
start_str = "2020-05-23 00:00:00"
end_str = "2020-05-23 00:30:00"
start_dt = pd.to_datetime(start_str).tz_localize(pytz.timezone('Asia/Shanghai'))
end_dt = pd.to_datetime(end_str).tz_localize(pytz.timezone('Asia/Shanghai'))

# ----- METRIC ANALYSIS -----
# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using entire series
metric_thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .agg(global_p95=lambda s: s.quantile(0.95),
         global_p5 = lambda s: s.quantile(0.05))
)

# 2) Filter to incident window
df_metric_window = df_metric[(df_metric['ts_dt'] >= start_dt) & (df_metric['ts_dt'] <= end_dt)].copy()

# 3) For each series compute requested in-window stats
# Merge thresholds into window rows so we can identify anomalies
metric_win_merged = df_metric_window.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')

# Flag anomalies
metric_win_merged['gt_p95'] = metric_win_merged['value'] > metric_win_merged['global_p95']
metric_win_merged['lt_p5'] = metric_win_merged['value'] < metric_win_merged['global_p5']
metric_win_merged['anomaly'] = metric_win_merged['gt_p95'] | metric_win_merged['lt_p5']

# Aggregate in-window per series
metric_inwindow_agg = (
    metric_win_merged
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(points_in_window = ('value','count'),
         count_gt_p95 = ('gt_p95','sum'),
         count_lt_p5 = ('lt_p5','sum'),
         earliest_anom_ts = ('ts_dt', lambda s: s[metric_win_merged.loc[s.index, 'anomaly']].min() if len(s)>0 else pd.NaT),
         max_in_window = ('value','max'),
         min_in_window = ('value','min'))
)

# Note: the lambda above uses index alignment to pick earliest anomaly timestamp.
# However groupby+agg lambda mapping of indices can be tricky; do a safe separate computation:
if metric_inwindow_agg['earliest_anom_ts'].isnull().any():
    # recompute earliest anomaly ts robustly
    earliest = (
        metric_win_merged[metric_win_merged['anomaly']]
        .groupby(['cmdb_id','kpi_name'], as_index=False)['ts_dt'].min()
        .rename(columns={'ts_dt':'earliest_anom_ts'})
    )
    metric_inwindow_agg = metric_inwindow_agg.drop(columns=['earliest_anom_ts']).merge(earliest, on=['cmdb_id','kpi_name'], how='left')

# Merge thresholds with in-window aggregated stats to form full per-series table
metric_series_full = metric_thresholds.merge(metric_inwindow_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for series with no in-window points
metric_series_full['points_in_window'] = metric_series_full['points_in_window'].fillna(0).astype(int)
metric_series_full['count_gt_p95'] = metric_series_full['count_gt_p95'].fillna(0).astype(int)
metric_series_full['count_lt_p5'] = metric_series_full['count_lt_p5'].fillna(0).astype(int)
metric_series_full['max_in_window'] = metric_series_full['max_in_window']
metric_series_full['min_in_window'] = metric_series_full['min_in_window']

# Compute max_exceed_ratio where applicable
def compute_ratio(row):
    p95 = row['global_p95']
    mx = row['max_in_window']
    if pd.isna(mx) or pd.isna(p95):
        return np.nan
    if p95 > 0:
        return (mx - p95) / p95
    return np.nan

metric_series_full['max_exceed_ratio'] = metric_series_full.apply(compute_ratio, axis=1)

# Exclude series with zero anomalies (count_gt_p95==0 and count_lt_p5==0)
metric_series_candidates = metric_series_full[ (metric_series_full['count_gt_p95']>0) | (metric_series_full['count_lt_p5']>0) ].copy()

# Sort by max(count_gt_p95, count_lt_p5) descending
metric_series_candidates['anomaly_count_max'] = metric_series_candidates[['count_gt_p95','count_lt_p5']].max(axis=1)
metric_anomalies_top20 = (
    metric_series_candidates
    .sort_values(['anomaly_count_max','points_in_window'], ascending=[False, False])
    .loc[:, ['cmdb_id','kpi_name','global_p95','global_p5','points_in_window','count_gt_p95','count_lt_p5','earliest_anom_ts','max_in_window','min_in_window','max_exceed_ratio']]
    .head(20)
    .reset_index(drop=True)
)

# ----- TRACE ANALYSIS (same procedure) -----
trace_thresholds = (
    df_trace
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(global_p95 = lambda s: s.quantile(0.95),
         global_p5  = lambda s: s.quantile(0.05))
)

df_trace_window = df_trace[(df_trace['ts_dt'] >= start_dt) & (df_trace['ts_dt'] <= end_dt)].copy()
trace_win_merged = df_trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
trace_win_merged['gt_p95'] = trace_win_merged['value'] > trace_win_merged['global_p95']
trace_win_merged['lt_p5'] = trace_win_merged['value'] < trace_win_merged['global_p5']
trace_win_merged['anomaly'] = trace_win_merged['gt_p95'] | trace_win_merged['lt_p5']

trace_inwindow_agg = (
    trace_win_merged
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(points_in_window = ('value','count'),
         count_gt_p95 = ('gt_p95','sum'),
         count_lt_p5 = ('lt_p5','sum'))
)

# earliest anomaly timestamp for traces
earliest_trace = (
    trace_win_merged[trace_win_merged['anomaly']]
    .groupby(['cmdb_id','trace_name'], as_index=False)['ts_dt'].min()
    .rename(columns={'ts_dt':'earliest_anom_ts'})
)
trace_inwindow_agg = trace_inwindow_agg.merge(earliest_trace, on=['cmdb_id','trace_name'], how='left')

trace_series_full = trace_thresholds.merge(trace_inwindow_agg, on=['cmdb_id','trace_name'], how='left')
trace_series_full['points_in_window'] = trace_series_full['points_in_window'].fillna(0).astype(int)
trace_series_full['count_gt_p95'] = trace_series_full['count_gt_p95'].fillna(0).astype(int)
trace_series_full['count_lt_p5'] = trace_series_full['count_lt_p5'].fillna(0).astype(int)

# Compute max/min in window for traces
trace_stats_window = (
    df_trace_window
    .groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(max_in_window=('value','max'), min_in_window=('value','min'))
)
trace_series_full = trace_series_full.merge(trace_stats_window, on=['cmdb_id','trace_name'], how='left')

def compute_ratio_trace(row):
    p95 = row['global_p95']
    mx = row['max_in_window']
    if pd.isna(mx) or pd.isna(p95):
        return np.nan
    if p95 > 0:
        return (mx - p95) / p95
    return np.nan

trace_series_full['max_exceed_ratio'] = trace_series_full.apply(compute_ratio_trace, axis=1)

# Filter candidate trace series with anomalies
trace_series_candidates = trace_series_full[(trace_series_full['count_gt_p95']>0) | (trace_series_full['count_lt_p5']>0)].copy()
trace_series_candidates['anomaly_count_max'] = trace_series_candidates[['count_gt_p95','count_lt_p5']].max(axis=1)
trace_anomalies_top20 = (
    trace_series_candidates
    .sort_values(['anomaly_count_max','points_in_window'], ascending=[False, False])
    .loc[:, ['cmdb_id','trace_name','global_p95','global_p5','points_in_window','count_gt_p95','count_lt_p5','earliest_anom_ts','max_in_window','min_in_window','max_exceed_ratio']]
    .head(20)
    .reset_index(drop=True)
)

# Keep full intermediate tables in kernel variables (do not print them)
metric_thresholds_table = metric_thresholds
metric_win_table = metric_win_merged
metric_series_full_table = metric_series_full

trace_thresholds_table = trace_thresholds
trace_win_table = trace_win_merged
trace_series_full_table = trace_series_full

# Final compact outputs (variables displayed)
metric_anomalies_top20, trace_anomalies_top20

```
Out[2]:
```
```python
# Summarize the anomaly analysis results in plain English as a compact string.
# Reuse variables from previous computation: metric_anomalies_top20, trace_anomalies_top20

# Prepare compact descriptions for top 5 metric anomalies
def compact_metric_rows(df, n=5):
    lines = []
    for r in df.head(n).itertuples(index=False):
        cmdb = r.cmdb_id
        kpi = r.kpi_name
        gt = int(r.count_gt_p95)
        lt = int(r.count_lt_p5)
        ts = pd.NaT if pd.isna(r.earliest_anom_ts) else str(r.earliest_anom_ts)
        mx = r.max_in_window if not pd.isna(r.max_in_window) else "n/a"
        ratio = r.max_exceed_ratio
        ratio_s = f"{ratio:.3f}" if (not pd.isna(ratio)) else "n/a"
        lines.append(f"{cmdb} | {kpi} | >P95:{gt} <P5:{lt} earliest:{ts} max:{mx} ratio:{ratio_s}")
    return " ; ".join(lines)

def compact_trace_rows(df, n=5):
    lines = []
    for r in df.head(n).itertuples(index=False):
        cmdb = r.cmdb_id
        name = r.trace_name
        gt = int(r.count_gt_p95)
        lt = int(r.count_lt_p5)
        ts = pd.NaT if pd.isna(r.earliest_anom_ts) else str(r.earliest_anom_ts)
        mx = r.max_in_window if not pd.isna(r.max_in_window) else "n/a"
        ratio = r.max_exceed_ratio
        ratio_s = f"{ratio:.3f}" if (not pd.isna(ratio)) else "n/a"
        lines.append(f"{cmdb} | {name} | >P95:{gt} <P5:{lt} earliest:{ts} max:{mx} ratio:{ratio_s}")
    return " ; ".join(lines)

metric_top5_text = compact_metric_rows(metric_anomalies_top20, n=5)
trace_top5_text = compact_trace_rows(trace_anomalies_top20, n=5)

summary = (
    "Anomaly analysis summary (incident window 2020-05-23 00:00:00+08 to 2020-05-23 00:30:00+08):\n\n"
    f"Metrics: {metric_anomalies_top20.shape[0]} series with anomalies (showing top {min(5, metric_anomalies_top20.shape[0])}):\n"
    f"{metric_top5_text}\n\n"
    f"Traces: {trace_anomalies_top20.shape[0]} series with anomalies (showing top {min(5, trace_anomalies_top20.shape[0])}):\n"
    f"{trace_top5_text}\n\n"
    "Notes: counts refer to points exceeding global P95 or below global P5 computed from the entire series. "
    "max_exceed_ratio = (max_in_window - global_p95) / global_p95 when global_p95>0."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                  kpi_name  global_p95  global_p5  points_in_window  count_gt_p95  count_lt_p5          earliest_anom_ts  max_in_window  min_in_window  max_exceed_ratio
0    db_001              CPU_free_pct   98.933205  97.749308                31             2            2 2020-05-23 00:09:00+08:00      99.045504      97.728462          0.001135
1    db_003              CPU_free_pct   97.808950  95.982407                31             2            2 2020-05-23 00:01:00+08:00      97.938080      94.962713          0.001320
2    db_008              CPU_free_pct   97.641676  96.294904                31             2            2 2020-05-23 00:07:00+08:00      97.950711      96.253908          0.003165
3    os_006                Send_total    0.000472   0.000309                31             2            2 2020-05-23 00:02:00+08:00       0.000821       0.000239          0.739407
4    os_008                Send_total    0.000681   0.000507                31             2            2 2020-05-23 00:09:00+08:00       0.001458       0.000350          1.142542
5    os_012              Disk_io_util    1.016838   0.000000                31             2            0 2020-05-23 00:02:00+08:00       2.000000       0.000000          0.966882
6    os_012      Processor_load_5_min    0.022500   0.005000                31             2            0 2020-05-23 00:05:00+08:00       0.026667       0.005000          0.185200
7    os_015  Incoming_network_traffic    0.003465   0.000000                31             2            0 2020-05-23 00:13:00+08:00       0.003550       0.000000          0.024383
8    os_017                Send_total    0.262525   0.062831                31             2            2 2020-05-23 00:00:00+08:00       0.264189       0.003984          0.006340
9    os_022  Outgoing_network_traffic    0.065787   0.018442                31             2            2 2020-05-23 00:00:00+08:00       0.068697       0.003247          0.044226
10  osb_001              app.avg_time    0.009375   0.000601                31             2            2 2020-05-23 00:07:00+08:00       0.036529       0.000569          2.896505
11  osb_001                   app.num  495.500000  84.750000                31             2            2 2020-05-23 00:07:00+08:00     499.000000      45.000000          0.007064
12  osb_001            app.succee_num  495.500000  84.750000                31             2            2 2020-05-23 00:07:00+08:00     499.000000      43.000000          0.007064
13  osb_001           app.succee_rate    1.000000   0.980000                31             0            2 2020-05-23 00:09:00+08:00       1.000000       0.800000          0.000000
14   db_001                       ACS    0.020000   0.004500                30             0            2 2020-05-23 00:18:00+08:00       0.020000       0.000000          0.000000
15   db_001              Call_Per_Sec   11.355000  10.700000                30             2            1 2020-05-23 00:01:00+08:00      12.600000      10.600000          0.109643
16   db_001                    DbTime   49.940000   5.100000                30             2            0 2020-05-23 00:19:00+08:00      59.600000       5.100000          0.193432
17   db_001              Exec_Per_Sec  190.895000   9.545000                30             2            2 2020-05-23 00:08:00+08:00     214.200000       9.500000          0.122083
18   db_001       LFParaWrite_Per_Sec   26.750500   0.000000                30             2            0 2020-05-23 00:13:00+08:00     152.100000       0.000000          4.685875
19   db_001            LFSync_Per_Sec    2.618500   0.000000                30             2            0 2020-05-23 00:00:00+08:00     181.290000       0.000000         68.234294,        cmdb_id                                       trace_name   global_p95   global_p5  points_in_window  count_gt_p95  count_lt_p5          earliest_anom_ts  max_in_window  min_in_window  max_exceed_ratio
0   docker_001    trace.from_os_022.remoteprocess.duration_mean     0.884476    0.269152                30             2            2 2020-05-23 00:05:00+08:00       1.276065       0.223452          0.442735
1   docker_001     trace.from_os_022.remoteprocess.duration_p95     2.880257    0.607855                30             2            2 2020-05-23 00:01:00+08:00       3.285550       0.411650          0.140714
2   docker_001        trace.from_os_022.remoteprocess.row_count   132.100000   35.450000                30             2            2 2020-05-23 00:06:00+08:00     133.000000      20.000000          0.006813
3   docker_001                     trace.self.csf.duration_mean     0.068302    0.031900                30             2            2 2020-05-23 00:05:00+08:00       0.088964       0.029343          0.302504
4   docker_001                      trace.self.csf.duration_p95     0.196483    0.062400                30             2            2 2020-05-23 00:05:00+08:00       0.197250       0.038000          0.003906
5   docker_001                         trace.self.csf.row_count   528.400000  141.800000                30             2            2 2020-05-23 00:06:00+08:00     532.000000      80.000000          0.006813
6   docker_001               trace.self.flyremote.duration_mean     0.012428    0.006032                30             2            2 2020-05-23 00:06:00+08:00       0.013783       0.005400          0.109046
7   docker_001                trace.self.flyremote.duration_p95     0.090210    0.007472                30             2            2 2020-05-23 00:06:00+08:00       0.094000       0.006300          0.042013
8   docker_001                   trace.self.flyremote.row_count   132.100000   35.450000                30             2            2 2020-05-23 00:06:00+08:00     133.000000      20.000000          0.006813
9   docker_001                    trace.self.jdbc.duration_mean     0.004291    0.001923                30             2            2 2020-05-23 00:05:00+08:00       0.005489       0.001786          0.279114
10  docker_001                     trace.self.jdbc.duration_p95     0.009042    0.003000                30             2            0 2020-05-23 00:04:00+08:00       0.070450       0.003000          6.790987
11  docker_001                        trace.self.jdbc.row_count  2116.500000  567.200000                30             2            2 2020-05-23 00:06:00+08:00    2131.000000     320.000000          0.006851
12  docker_001                   trace.self.local.duration_mean     0.020445    0.004985                30             2            2 2020-05-23 00:05:00+08:00       0.031274       0.004143          0.529628
13  docker_001                    trace.self.local.duration_p95     0.098550    0.008000                30             2            0 2020-05-23 00:27:00+08:00       0.100000       0.008000          0.014713
14  docker_001                       trace.self.local.row_count  1849.400000  496.300000                30             2            2 2020-05-23 00:06:00+08:00    1862.000000     280.000000          0.006813
15  docker_001  trace.to_docker_007.remoteprocess.duration_mean     0.025885    0.024014                30             2            2 2020-05-23 00:06:00+08:00       0.026116       0.022932          0.008936
16  docker_001   trace.to_docker_007.remoteprocess.duration_p95     0.037005    0.033000                30             2            1 2020-05-23 00:06:00+08:00       0.039300       0.032000          0.062019
17  docker_001      trace.to_docker_007.remoteprocess.row_count   258.650000   72.450000                30             2            2 2020-05-23 00:08:00+08:00     276.000000      39.000000          0.067079
18  docker_001  trace.to_docker_008.remoteprocess.duration_mean     0.026195    0.023515                30             2            2 2020-05-23 00:03:00+08:00       0.027637       0.023220          0.055035
19  docker_001   trace.to_docker_008.remoteprocess.duration_p95     0.036843    0.033000                30             2            0 2020-05-23 00:03:00+08:00       0.038550       0.033000          0.046346)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric and tz if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Parameters
cmdb = 'db_001'
kpis = ["LFSync_Per_Sec", "LFParaWrite_Per_Sec", "DbTime", "Exec_Per_Sec", "Call_Per_Sec", "ACS"]
start_dt = pd.to_datetime("2020-05-23 00:00:00").tz_localize(tz)
end_dt   = pd.to_datetime("2020-05-23 00:30:00").tz_localize(tz)

# 1) Compute global P95 and P5 per KPI for db_001 using full series (before filtering)
metric_db = df_metric[df_metric['cmdb_id'] == cmdb].copy()
metric_db_kpis = metric_db[metric_db['kpi_name'].isin(kpis)].copy()

metric_db_thresholds = (
    metric_db_kpis
    .groupby('kpi_name', as_index=False)['value']
    .agg(global_p95=lambda s: s.quantile(0.95),
         global_p5 =lambda s: s.quantile(0.05))
)

# 2) Filter to incident window for db_001 and selected KPIs
df_db_window = metric_db_kpis[(metric_db_kpis['ts_dt'] >= start_dt) & (metric_db_kpis['ts_dt'] <= end_dt)].copy()

# Merge thresholds into windowed rows for breach detection
db_win_merged = df_db_window.merge(metric_db_thresholds, on='kpi_name', how='left')

# Compute breach booleans
db_win_merged['gt_p95'] = db_win_merged['value'] > db_win_merged['global_p95']
db_win_merged['lt_p5'] = db_win_merged['value'] < db_win_merged['global_p5']
db_win_merged['breach'] = db_win_merged['gt_p95'] | db_win_merged['lt_p5']

# Helper: compute longest consecutive breach run length for a boolean Series sorted by timestamp
def longest_consecutive_run_bool(s_bool):
    if s_bool.empty:
        return 0
    grp = s_bool.ne(s_bool.shift()).cumsum()
    lengths = s_bool.groupby(grp).size()
    first_vals = s_bool.groupby(grp).first()
    breach_lengths = lengths[first_vals==True]
    if breach_lengths.empty:
        return 0
    return int(breach_lengths.max())

# Aggregate per KPI for summary
summary_rows = []
for k in kpis:
    # thresholds
    thr_row = metric_db_thresholds[metric_db_thresholds['kpi_name']==k]
    if thr_row.empty:
        p95 = np.nan
        p5 = np.nan
    else:
        p95 = float(thr_row['global_p95'].iloc[0])
        p5  = float(thr_row['global_p5'].iloc[0])
    # in-window rows for this KPI
    df_k = db_win_merged[db_win_merged['kpi_name']==k].sort_values('ts_dt').copy()
    points_in_window = int(df_k.shape[0])
    count_gt_p95 = int(df_k['gt_p95'].sum()) if points_in_window>0 else 0
    count_lt_p5  = int(df_k['lt_p5'].sum()) if points_in_window>0 else 0
    if points_in_window>0 and df_k['breach'].any():
        earliest_breach_ts = df_k.loc[df_k['breach'], 'ts_dt'].min()
    else:
        earliest_breach_ts = pd.NaT
    max_in_window = df_k['value'].max() if points_in_window>0 else np.nan
    min_in_window = df_k['value'].min() if points_in_window>0 else np.nan
    # longest consecutive breach run (using the boolean sequence ordered by ts)
    if points_in_window>0:
        longest_run = longest_consecutive_run_bool(df_k['breach'].reset_index(drop=True))
    else:
        longest_run = 0
    # max_exceed_ratio
    if (not pd.isna(p95)) and (p95 > 0) and (not pd.isna(max_in_window)):
        max_exceed_ratio = (max_in_window - p95) / p95
    else:
        max_exceed_ratio = np.nan
    summary_rows.append({
        'kpi_name': k,
        'global_p95': p95,
        'global_p5': p5,
        'points_in_window': points_in_window,
        'count_gt_p95': count_gt_p95,
        'count_lt_p5': count_lt_p5,
        'earliest_breach_ts': earliest_breach_ts,
        'longest_consecutive_breach_length': longest_run,
        'max_in_window': max_in_window,
        'min_in_window': min_in_window,
        'max_exceed_ratio': max_exceed_ratio
    })

summary_df = pd.DataFrame(summary_rows)

# 3) Prepare raw in-window rows per KPI limited to 31 rows, with breach_type column
raw_rows_list = []
for k in kpis:
    thr_row = metric_db_thresholds[metric_db_thresholds['kpi_name']==k]
    if thr_row.empty:
        p95 = np.nan
        p5 = np.nan
    else:
        p95 = float(thr_row['global_p95'].iloc[0])
        p5  = float(thr_row['global_p5'].iloc[0])
    df_k = df_db_window[df_db_window['kpi_name']==k].sort_values('ts_dt').copy()
    if df_k.empty:
        continue
    # determine breach_type
    def btype(v):
        if (not pd.isna(p95)) and v > p95:
            return 'gt_p95'
        if (not pd.isna(p5)) and v < p5:
            return 'lt_p5'
        return 'normal'
    df_k['breach_type'] = df_k['value'].apply(btype)
    df_k_limited = df_k[['timestamp','ts_dt','cmdb_id','kpi_name','value','breach_type']].head(31).copy()
    raw_rows_list.append(df_k_limited)

# Concatenate raw rows (this will include up to 31 rows per KPI)
if raw_rows_list:
    raw_rows_db001 = pd.concat(raw_rows_list, ignore_index=True).sort_values(['kpi_name','ts_dt']).reset_index(drop=True)
else:
    raw_rows_db001 = pd.DataFrame(columns=['timestamp','ts_dt','cmdb_id','kpi_name','value','breach_type'])

# Store full intermediate tables in kernel variables (do not print)
metric_db_thresholds_table = metric_db_thresholds
db_window_table = df_db_window
db_win_merged_table = db_win_merged
summary_full_table = summary_df
raw_rows_full_table = raw_rows_db001

# Final compact outputs: the summary (one row per KPI) and the per-KPI in-window rows (<=31 each)
summary_df, raw_rows_db001

```
Out[3]:
```
```python
# Summarize results for db_001 KPIs in plain English using existing kernel variables.
# Reuse summary_df from prior execution which contains the per-KPI summary.

from math import isnan
import pandas as pd

try:
    summary_df
except NameError:
    raise RuntimeError("summary_df not found in kernel. Run the analysis code first.")

lines = []
lines.append("db_001 — KPI anomaly summary for window 2020-05-23 00:00:00+08 to 2020-05-23 00:30:00+08:\n")

def fmt(x, nd=6):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return "null"
    if isinstance(x, pd.Timestamp):
        return str(x)
    if isinstance(x, float):
        # compact numeric formatting
        return f"{x:.6g}"
    return str(x)

for r in summary_df.itertuples(index=False):
    kp = r.kpi_name
    p95 = fmt(r.global_p95, nd=6)
    p5  = fmt(r.global_p5, nd=6)
    pts = int(r.points_in_window)
    gt  = int(r.count_gt_p95)
    lt  = int(r.count_lt_p5)
    eb  = fmt(r.earliest_breach_ts)
    longest = int(r.longest_consecutive_breach_length)
    mx = fmt(r.max_in_window)
    mn = fmt(r.min_in_window)
    ratio = ("null" if (r.max_exceed_ratio is None or (isinstance(r.max_exceed_ratio,float) and pd.isna(r.max_exceed_ratio)))
             else f"{r.max_exceed_ratio:.6g}")
    lines.append(f"- {kp}: global_P95={p95}, global_P5={p5}; points_in_window={pts}; >P95={gt}, <P5={lt}; "
                 f"earliest_breach={eb}; longest_consecutive_breach={longest}; max={mx}, min={mn}; max_exceed_ratio={ratio}")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(              kpi_name  global_p95  global_p5  points_in_window  count_gt_p95  count_lt_p5        earliest_breach_ts  longest_consecutive_breach_length  max_in_window  min_in_window  max_exceed_ratio
0       LFSync_Per_Sec      2.6185     0.0000                30             2            0 2020-05-23 00:00:00+08:00                                  1         181.29            0.0         68.234294
1  LFParaWrite_Per_Sec     26.7505     0.0000                30             2            0 2020-05-23 00:13:00+08:00                                  1         152.10            0.0          4.685875
2               DbTime     49.9400     5.1000                30             2            0 2020-05-23 00:19:00+08:00                                  1          59.60            5.1          0.193432
3         Exec_Per_Sec    190.8950     9.5450                30             2            2 2020-05-23 00:08:00+08:00                                  1         214.20            9.5          0.122083
4         Call_Per_Sec     11.3550    10.7000                30             2            1 2020-05-23 00:01:00+08:00                                  1          12.60           10.6          0.109643
5                  ACS      0.0200     0.0045                30             0            2 2020-05-23 00:18:00+08:00                                  1           0.02            0.0          0.000000,       timestamp                     ts_dt cmdb_id        kpi_name  value breach_type
0    1590163200 2020-05-23 00:00:00+08:00  db_001             ACS   0.01      normal
1    1590163260 2020-05-23 00:01:00+08:00  db_001             ACS   0.01      normal
2    1590163320 2020-05-23 00:02:00+08:00  db_001             ACS   0.02      normal
3    1590163380 2020-05-23 00:03:00+08:00  db_001             ACS   0.02      normal
4    1590163440 2020-05-23 00:04:00+08:00  db_001             ACS   0.01      normal
..          ...                       ...     ...             ...    ...         ...
175  1590164700 2020-05-23 00:25:00+08:00  db_001  LFSync_Per_Sec   1.69      normal
176  1590164760 2020-05-23 00:26:00+08:00  db_001  LFSync_Per_Sec   0.00      normal
177  1590164820 2020-05-23 00:27:00+08:00  db_001  LFSync_Per_Sec   1.75      normal
178  1590164880 2020-05-23 00:28:00+08:00  db_001  LFSync_Per_Sec   0.00      normal
179  1590164940 2020-05-23 00:29:00+08:00  db_001  LFSync_Per_Sec   1.44      normal

[180 rows x 6 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric and tz if available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['ts_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Parameters
cmdb = 'db_001'
kpi = 'LFSync_Per_Sec'
start_dt = pd.to_datetime("2020-05-23 00:00:00").tz_localize(tz)
end_dt   = pd.to_datetime("2020-05-23 00:30:00").tz_localize(tz)

# 1) Compute global P95 and P5 using all db_001 LFSync_Per_Sec rows (before window filter)
df_k_all = df_metric[(df_metric['cmdb_id']==cmdb) & (df_metric['kpi_name']==kpi)].copy()
if df_k_all.empty:
    global_p95 = np.nan
    global_p5  = np.nan
else:
    global_p95 = float(df_k_all['value'].quantile(0.95))
    global_p5  = float(df_k_all['value'].quantile(0.05))

# 2) Filter to incident window and prepare rows with breach_flag
df_k_window = df_k_all[(df_k_all['ts_dt'] >= start_dt) & (df_k_all['ts_dt'] <= end_dt)].sort_values('ts_dt').copy()

def determine_flag(v, p95, p5):
    if (not pd.isna(p95)) and v > p95:
        return 'gt_p95'
    if (not pd.isna(p5)) and v < p5:
        return 'lt_p5'
    return 'normal'

df_k_window['breach_flag'] = df_k_window['value'].apply(lambda v: determine_flag(v, global_p95, global_p5))

# Limit raw rows to at most 31 rows sorted by timestamp
raw_rows_limited = df_k_window[['timestamp','ts_dt','cmdb_id','kpi_name','value','breach_flag']].head(31).reset_index(drop=True)

# 3) Tiny summary: first_breach_ts, total_breach_points, longest_consecutive_breach_run_length
if df_k_window.empty:
    first_breach_ts = pd.NaT
    total_breach_points = 0
    longest_run = 0
else:
    breaches_bool = df_k_window['breach_flag'] != 'normal'
    total_breach_points = int(breaches_bool.sum())
    first_breach_ts = df_k_window.loc[breaches_bool, 'ts_dt'].min() if total_breach_points>0 else pd.NaT
    # compute longest consecutive run of True in breaches_bool ordered by ts_dt
    if breaches_bool.empty or (~breaches_bool).all():
        longest_run = 0
    else:
        grp = breaches_bool.ne(breaches_bool.shift()).cumsum()
        lengths = breaches_bool.groupby(grp).size()
        vals = breaches_bool.groupby(grp).first()
        breach_lengths = lengths[vals==True]
        longest_run = int(breach_lengths.max()) if not breach_lengths.empty else 0

summary_small = pd.DataFrame([{
    'kpi_name': kpi,
    'global_p95': global_p95,
    'global_p5': global_p5,
    'points_in_window': int(df_k_window.shape[0]),
    'first_breach_ts': first_breach_ts,
    'total_breach_points': total_breach_points,
    'longest_consecutive_breach_run_length': longest_run
}])

# Keep full intermediate tables in kernel variables (do not print them)
df_k_all_table = df_k_all
df_k_window_table = df_k_window

# Final compact outputs (display summary and up to 31 raw rows)
summary_small, raw_rows_limited

```
Out[4]:
```
```python
# Summarize findings for db_001 / LFSync_Per_Sec using existing kernel variables.
# Reuse: summary_small, raw_rows_limited, df_k_window_table (created in previous cell)

import pandas as pd

# Extract values from summary_small
row = summary_small.iloc[0]
global_p95 = row['global_p95']
global_p5  = row['global_p5']
points_in_window = int(row['points_in_window'])
first_breach_ts = row['first_breach_ts']
total_breach_points = int(row['total_breach_points'])
longest_run = int(row['longest_consecutive_breach_run_length'])

# Additional stats from full in-window table if available
try:
    df_win = df_k_window_table  # full in-window df for this KPI
    max_in_window = df_win['value'].max() if not df_win.empty else None
    min_in_window = df_win['value'].min() if not df_win.empty else None
    # list breach occurrences in-window (timestamp, value, flag) from limited raw rows for compactness
    breaches = raw_rows_limited[raw_rows_limited['breach_flag'] != 'normal'][['ts_dt','value','breach_flag']]
    breaches_list = [f"{r.ts_dt} ({r.breach_flag}, value={r.value})" for r in breaches.itertuples(index=False)]
except NameError:
    max_in_window = None
    min_in_window = None
    breaches_list = []

summary_text = (
    f"db_001 / LFSync_Per_Sec summary (global thresholds computed from full series):\n"
    f"- global P95 = {global_p95}, global P5 = {global_p5}\n"
    f"- incident window 2020-05-23 00:00:00+08 to 2020-05-23 00:30:00+08: {points_in_window} points\n"
    f"- total breach points in-window: {total_breach_points} (breach type: {'gt_p95' if total_breach_points>0 else 'none'})\n"
    f"- first breach timestamp in-window: {first_breach_ts}\n"
    f"- longest consecutive breach run length: {longest_run}\n"
    f"- in-window max = {max_in_window}, min = {min_in_window}\n"
    f"- breach occurrences (compact): " + (", ".join(breaches_list) if breaches_list else "none") + "\n\n"
    f"Raw in-window rows (up to 31) are available in variable 'raw_rows_limited' (columns: timestamp, ts_dt, cmdb_id, kpi_name, value, breach_flag)."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

(         kpi_name  global_p95  global_p5  points_in_window           first_breach_ts  total_breach_points  longest_consecutive_breach_run_length
0  LFSync_Per_Sec      2.6185        0.0                30 2020-05-23 00:00:00+08:00                    2                                      1,      timestamp                     ts_dt cmdb_id        kpi_name  value breach_flag
0   1590163200 2020-05-23 00:00:00+08:00  db_001  LFSync_Per_Sec   2.65      gt_p95
1   1590163260 2020-05-23 00:01:00+08:00  db_001  LFSync_Per_Sec   1.86      normal
2   1590163320 2020-05-23 00:02:00+08:00  db_001  LFSync_Per_Sec   2.58      normal
3   1590163380 2020-05-23 00:03:00+08:00  db_001  LFSync_Per_Sec   1.86      normal
4   1590163440 2020-05-23 00:04:00+08:00  db_001  LFSync_Per_Sec   0.00      normal
..         ...                       ...     ...             ...    ...         ...
25  1590164700 2020-05-23 00:25:00+08:00  db_001  LFSync_Per_Sec   1.69      normal
26  1590164760 2020-05-23 00:26:00+08:00  db_001  LFSync_Per_Sec   0.00      normal
27  1590164820 2020-05-23 00:27:00+08:00  db_001  LFSync_Per_Sec   1.75      normal
28  1590164880 2020-05-23 00:28:00+08:00  db_001  LFSync_Per_Sec   0.00      normal
29  1590164940 2020-05-23 00:29:00+08:00  db_001  LFSync_Per_Sec   1.44      normal

[30 rows x 6 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```